<h1 style="color:DodgerBlue">Индивидуальный проект</h1>
<h2 style="color:DodgerBlue">Вариант задания №12</h2>

**Студент:** Лаврентьев Владислав  
**Возраст:** 19 лет  
**Группа:** ИСПБ-25-1, Тюмень  

Создать базовый класс Item для товаров, которые могут быть заказаны или возвращены. На его основе реализовать производные классы, демонстрирующие наследование и полиморфизм.

Для скидки используется процент от текущей цены: например, `10` означает скидку 10%.

<h2 style="color:DodgerBlue">Реализация</h2>

Базовый класс хранит идентификатор, название и цену. Производные классы добавляют собственные атрибуты или изменяют поведение методов.

In [ ]:
class Item
{
    public int ItemId { get; }
    public string Name { get; }
    public decimal Price { get; protected set; }

    public Item(int itemId, string name, decimal price)
    {
        if (itemId <= 0) throw new ArgumentOutOfRangeException(nameof(itemId));
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Название не может быть пустым", nameof(name));
        if (price < 0) throw new ArgumentOutOfRangeException(nameof(price));

        ItemId = itemId;
        Name = name;
        Price = price;
    }

    public virtual string GetDetails() => $"Товар №{ItemId}: {Name}, цена {Price:C}";

    public virtual decimal CalculateDiscount() => Price * 0.05m;

    public virtual void ApplyDiscount(decimal discount)
    {
        if (discount < 0 || discount > 100) throw new ArgumentOutOfRangeException(nameof(discount));
        Price *= 1 - discount / 100;
    }
}

class SingleItem : Item
{
    public string UnitMeasure { get; }

    public SingleItem(int itemId, string name, decimal price, string unitMeasure)
        : base(itemId, name, price) => UnitMeasure = unitMeasure;

    public override string GetDetails() => $"{base.GetDetails()}, единица измерения: {UnitMeasure}";
}

class PackageItem : Item
{
    public int QuantityPerPackage { get; }

    public PackageItem(int itemId, string name, decimal price, int quantityPerPackage)
        : base(itemId, name, price)
    {
        if (quantityPerPackage <= 0) throw new ArgumentOutOfRangeException(nameof(quantityPerPackage));
        QuantityPerPackage = quantityPerPackage;
    }

    public override string GetDetails() => $"{base.GetDetails()}, единиц в упаковке: {QuantityPerPackage}";

    public override decimal CalculateDiscount() => Price * 0.05m * QuantityPerPackage;
}

class SpecialItem : Item
{
    public DateTime DiscountExpirationDate { get; }

    public SpecialItem(int itemId, string name, decimal price, DateTime discountExpirationDate)
        : base(itemId, name, price) => DiscountExpirationDate = discountExpirationDate;

    public override string GetDetails() => $"{base.GetDetails()}, скидка действует до {DiscountExpirationDate:dd.MM.yyyy}";

    public override void ApplyDiscount(decimal discount)
    {
        if (DateTime.Today <= DiscountExpirationDate) base.ApplyDiscount(discount);
    }
}

class Program
{
    public static void Main()
    {
        Item[] items =
        {
            new SingleItem(1, "Молоко", 89.90m, "литр"),
            new PackageItem(2, "Чай", 350m, 20),
            new SpecialItem(3, "Кофе", 499m, DateTime.Today.AddDays(7))
        };

        foreach (Item item in items)
        {
            Console.WriteLine(item.GetDetails());
            Console.WriteLine($"Размер скидки: {item.CalculateDiscount():C}");
            item.ApplyDiscount(10);
            Console.WriteLine($"Цена после скидки 10%: {item.Price:C}");
            Console.WriteLine();
        }
    }
}